In [0]:
BASE_DIR = '/Volumes/sales/landing/volumes/Products/'
CHECKPOINT_DIR = '/Volumes/sales/landing/volumes/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/Products'
CATALOG = 'SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'PRODUCTS'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
products_schema = """product_id STRING, 
                 product_category_name STRING,
                 product_name_lenght DECIMAL(10,1),
                 product_description_lenght DECIMAL(10,1),
                 product_photos_qty DECIMAL(10,1),
                 product_weight_g DECIMAL(10,1),
                 product_length_cm DECIMAL(10,1),
                 product_height_cm DECIMAL(10,1),
                 product_width_cm DECIMAL(10,1)
                 """

In [0]:
products_df = (spark.read.format('csv').option('header','true').schema(products_schema).load(f'{BASE_DIR}')
           )
display(products_df.limit(3))
products_df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

products_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(products_schema)
                 .load(f'{BASE_DIR}')
        )
products_trans_df = products_df.select(
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

In [0]:
qry = (
    products_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()